# Run evaluation on datasets for non-standard speech

This includes reporting predictions and metrics.

Default approach for getting WER/CER on any model, base or fine-tuned. Run once per model (Baseline / Stage 1 / Stage 2) and once per normalization setting for the full before/after comparison.

For Kinyarwanda it includes apostrophe normalization (strips the elision apostrophe and joins tokens: n'abantu -> nabantu) as well as other transcript normalizations (both optional) to adjust for transcription inconsistencies.


# Configs

In [ ]:
# specify language: en or sw 
# Note: sw for Kinyarwanda, because we finetuned the Whisper model on Kinyarwanda data using Whisper's sw language code
LANGUAGE = "sw"
# LANGUAGE = "en"


if LANGUAGE == "en":
    DATASET_NAME = "cdli/rwandan_english_nonstandard_speech_v1.0"
    WHISPER_MODEL_NAME = "cdli/whisper-large-v3_finetuned_rwandan_english_nonstandard_speech_v1.0"

    # No apostrophe normalization for English.
    APPLY_KY_NORMALIZATION = False
    APPLY_KY_ADAPTED_WER = False
    
elif LANGUAGE == "sw":
    DATASET_NAME = "cdli/rwandan_kinyarwanda_nonstandard_speech_v1.0"
    
    # WHISPER_MODEL_NAME = "cdli/whisper-large-v3_finetuned_rwandan_kinyarwanda_standard_speech_v1.0"
    WHISPER_MODEL_NAME = "cdli/whisper-large-v3_finetuned_rwandan_kinyarwanda_nonstandard_speech_v1.0"

    # We need to apply special normalization for apostrophes in Kinyarwanda.
    APPLY_KY_NORMALIZATION = True   

    # Also apply additional normalization rules specific to Kinyarwanda for WER calculation.
    APPLY_KY_ADAPTED_WER = True

else:
    raise ValueError(f"Invalid language: {LANGUAGE}. Must be 'en' or 'sw' (sw for Kinyarwanda).")




# filter out utterances longer than 30 seconds (required for Whisper)
# should always be True
LENGTH_LIMIT = True


In [ ]:
# Because HF datasets have gated access, you need to request access first and then login to Hugging Face Hub to access the datasets.

from huggingface_hub import login
HF_TOKEN = input()
login(token=HF_TOKEN)

# Imports and defs

In [ ]:
# check GPU
!nvidia-smi

In [ ]:
import datasets
from IPython.display import Audio, display

import itables
itables.init_notebook_mode()
from huggingface_hub import hf_hub_download
import numpy as np
import pandas as pd
import os

import torch
from tqdm.auto import tqdm
from transformers.pipelines.pt_utils import Dataset

In [ ]:
# more efficient dataset handling
datasets.disable_caching()
print('cache:', datasets.is_caching_enabled())

torch.set_num_threads(1)
torch.get_num_threads()

In [ ]:
# note: this is slower when more stored in dataset (eg features), as it leads to data being copied around
class AudioTextDataset(Dataset):

    DEFAULT_UTTERANCE_FIELDS = ['audio_id', 'speaker_id',  'language', 'locale', 'accents',
                         'prompt_type', 'prompt_id',
                         'transcription', 'audio_length', 'transcript_length']

    def __init__(self, dataset: Dataset, utterance_fields=DEFAULT_UTTERANCE_FIELDS):
        self.dataset = dataset
        self.utterance_fields = utterance_fields

    def __len__(self):
        return len(self.dataset)


    def __getitem__(self, i):

        # only select the fields if they are present in the dataset
        result = {k: self.dataset[i][k] for k in self.utterance_fields if k in self.dataset[i]}
        # these two fields are required for the ASR pipeline
        result['sampling_rate'] = self.dataset[i]['audio']['sampling_rate']
        result['raw'] = self.dataset[i]['audio']['array']
        return result


In [ ]:
from evaluate import load as metrics_loader
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
import re

wer_metric = metrics_loader("wer")
cer_metric = metrics_loader("cer")

transcript_normalizer = BasicTextNormalizer()

# --- Kinyarwanda apostrophe normalization ---
# Strips the elision apostrophe and joins tokens: n'abantu -> nabantu.
# Assumes text is already glyph-normalized to the canonical ASCII apostrophe
CANONICAL_APOSTROPHE = "'"
_ELISION_RE = re.compile(r"\s*" + re.escape(CANONICAL_APOSTROPHE) + r"\s*")
_WS_RE = re.compile(r"\s+")

def normalize_ky_apostrophe(text: str) -> str:
    if not text:
        return text
    text = _ELISION_RE.sub("", text)
    text = _WS_RE.sub(" ", text).strip()
    return text



# --- transcription normalization for adapted WER for KY
# spelling/word-boundary variants agreed as "not a real error" ---
# Common and known meaning-changing pairs kept apart (eg kigali/kigari). 
# Repeated tokens never merged (verbatim stutter).
RL_KEEP = {"kigali", "kigari"}
E2I = set("parike serivise isahane sahane kwisahane nisahane kugasahane banke telefone "
          "piyese amarange emutiyene amagazine gaze zone esanse masaye dorone".split())
CY_KEEP = set("cyane cyo cya cyu icyo nicyo cyose cyanjye cyumuhondo".split())

RULES = {
    "ni + i elision": lambda t: re.sub(r"\bni i(?=[nm])", "ni ", t),
    "spacing": lambda t: re.sub(r"(?<!\bko)\s+ko\b", "ko", re.sub(r"\b(mu|ku|aba|umu|ni)\s+(?!\1\b)", r"\1", t)),
    "compound words": lambda t: t.replace("kugira ngo", "kugirango"),
    "r/l": lambda t: " ".join(w if w in RL_KEEP else w.replace("l", "r") for w in t.split()),
    "e/i loanword": lambda t: " ".join(w[:-1] + "i" if w in E2I else w for w in t.split()),
    "cy/k": lambda t: " ".join(w if w in CY_KEEP else w.replace("cy", "k") for w in t.split()),
    "jy/g": lambda t: " ".join(w.replace("jy", "g") if len(w) >= 5 else w for w in t.split()),
    "y-glide": lambda t: " ".join("zaria" if w == "zariya" else w for w in t.split()),
}

def normalize_adapted(text, patterns=RULES):
    for name in patterns:
        text = RULES[name](text)
    return _WS_RE.sub(" ", text).strip()


def normalize_text(text, adapted=False):
    if APPLY_KY_NORMALIZATION:
        text = normalize_ky_apostrophe(text)
    text = transcript_normalizer(text)
    return normalize_adapted(text) if adapted and APPLY_KY_ADAPTED_WER else text


# A prediction far longer than its reference is a decoding loop (model problem), not stutter.
def is_runaway(ref, pred, factor=2, slack=5):
    return len(pred.split()) > factor * len(ref.split()) + slack


def get_wer_cer(references, predictions,
                calculate_utterance_level_averaged_wer=False,
                normalize=True, adapted=False, verbose=True,
                ):
  # calculate_utterance_level_averaged_wer -- we first calculate the WER per
  # utterance and then average. This is not the standard way to calculate WER
  # on a corpus, but in a scenario of high WER (as for NSS) this allows to cap
  # at 1.0 on a per-utterance level.
  # adapted -- if True (and APPLY_KY_ADAPTED_WER is True), also fold in the
  # Kinyarwanda spelling/word-boundary rules above.
  predictions = list(predictions)
  references = list(references)
  if normalize:
    pred_strs = [normalize_text(x, adapted) for x in predictions]
    label_strs = [normalize_text(x, adapted) for x in references]
  else:
    pred_strs = list(predictions)
    label_strs = list(references)
  if calculate_utterance_level_averaged_wer:
    wers = []
    cers = []
    for pred_str, label_str in zip(pred_strs, label_strs):
      wer = wer_metric.compute(predictions=[pred_str], references=[label_str])
      cer = cer_metric.compute(predictions=[pred_str], references=[label_str])
      wers.append(wer)
      cers.append(cer)
      if verbose:
        print(label_str, '-->', pred_str, '-->', wer, cer)
    wer = np.mean([min(1.0, x) for x in wers])
    cer = np.mean([min(1.0, x) for x in cers])
  else:
    wer =  min(1, wer_metric.compute(references=label_strs, predictions=pred_strs))
    cer =  min(1, cer_metric.compute(references=label_strs, predictions=pred_strs))

    if verbose:
      for pred_str, label_str in zip(pred_strs, label_strs):
        print(label_str, '-->', pred_str)

  return (wer, cer)


In [ ]:
def prepare_results(results):
    df = pd.DataFrame(results)
    def clean_res(row):
        for f in df.columns:
            v = row[f]
            if isinstance(v, list):
                row[f] = v[0]
        return row
    df = df.apply(clean_res, axis=1)
    df['prediction'] = df['text'].str.strip() if isinstance(df['text'], str) else df['text']

    df['ground_truth'] = df['transcription'].str.strip() if isinstance(df['transcription'], str) else df['transcription']
    df = df.drop(columns=['text', 'transcription'])
    return df

def add_speaker_metadata(results_df, metadata_df):
    # Merge metadata with results_df on 'speaker_id'
    merged_df = results_df.merge(metadata_df, on='speaker_id', how='left')

    # Check if any speaker_id in results_df is missing in metadata_df
    missing_speakers = set(results_df['speaker_id']) - set(metadata_df['speaker_id'])
    if missing_speakers:
        print(f"Warning: Missing metadata for speakers: {missing_speakers}")

    return merged_df

def calculate_error_rates(results_df, verbose=False):

  def get_row_wer(row, adapted=False):
    return get_wer_cer(references=[row['ground_truth']], predictions=[row['prediction']], normalize=True, adapted=adapted, verbose=verbose)[0]

  def get_row_cer(row, adapted=False):
    return get_wer_cer(references=[row['ground_truth']], predictions=[row['prediction']], normalize=True, adapted=adapted, verbose=verbose)[1]




  # add normalized ground truth and prediction
  results_df['ground_truth_normalized'] = results_df['ground_truth'].apply(lambda x: transcript_normalizer(normalize_ky_apostrophe(x)) if APPLY_KY_NORMALIZATION else transcript_normalizer(x))
  results_df['prediction_normalized'] = results_df['prediction'].apply(lambda x: transcript_normalizer(normalize_ky_apostrophe(x)) if APPLY_KY_NORMALIZATION else transcript_normalizer(x))

  results_df['wer'] = results_df.apply(get_row_wer, axis=1)
  results_df['cer'] = results_df.apply(get_row_cer, axis=1)
  if APPLY_KY_ADAPTED_WER:
    results_df['wer_adapted'] = results_df.apply(lambda row: get_row_wer(row, adapted=True), axis=1)

  # decoding loops: a prediction far longer than its reference, not stutter (which stays verbatim)
  results_df['runaway'] = [is_runaway(r, p) for r, p in
                            zip(results_df['ground_truth_normalized'], results_df['prediction_normalized'])]
  n_loops = int(results_df['runaway'].sum())
  print(f'Decoding loops flagged: {n_loops} of {len(results_df)} utterances')

  overall_wer_cer = get_wer_cer(references=results_df.ground_truth.tolist(),
                                predictions=results_df.prediction.tolist(),
                                calculate_utterance_level_averaged_wer=False,
                                normalize=True, verbose=verbose)

  avg_utterance_level_wer_cer = get_wer_cer(references=results_df.ground_truth.tolist(),
                                predictions=results_df.prediction.tolist(),
                                calculate_utterance_level_averaged_wer=True,
                                normalize=True, verbose=verbose)


  print('Overall WER (normalized):', round(overall_wer_cer[0],3))
  print('Overall CER (normalized):', round(overall_wer_cer[1],3))
  print('Avg WER (normalized):', round(avg_utterance_level_wer_cer[0],3))
  print('Avg CER (normalized):', round(avg_utterance_level_wer_cer[1],3))

  if APPLY_KY_ADAPTED_WER:
    no_loops = results_df[~results_df['runaway']]
    overall_wer_adapted = get_wer_cer(references=results_df.ground_truth.tolist(),
                                      predictions=results_df.prediction.tolist(),
                                      normalize=True, adapted=True, verbose=verbose)
    overall_wer_noloops = get_wer_cer(references=no_loops.ground_truth.tolist(),
                                      predictions=no_loops.prediction.tolist(),
                                      normalize=True, verbose=verbose)
    overall_wer_adapted_noloops = get_wer_cer(references=no_loops.ground_truth.tolist(),
                                              predictions=no_loops.prediction.tolist(),
                                              normalize=True, adapted=True, verbose=verbose)
    print('Overall Adapted WER:', round(overall_wer_adapted[0],3))
    print('Overall WER (loops excluded):', round(overall_wer_noloops[0],3))
    print('Overall Adapted WER (loops excluded):', round(overall_wer_adapted_noloops[0],3))

  return results_df

In [ ]:
def load_dataset(dataset_name, limit_to_30_seconds=True):
    """
    Load a dataset from Hugging Face Hub.
    If limit_to_30_seconds is True, will only load examples with audio length <= 30 seconds.
    """
    ds = datasets.load_dataset(dataset_name, split='test', streaming=False)
    orig_len = len(ds)
    if limit_to_30_seconds:
        ds = ds.filter(lambda example: example['audio_length'] <= 30)
        print(f"Filtered dataset from {orig_len} to {len(ds)} examples with audio length <= 30 seconds")
    return ds

def load_speaker_metadata(dataset_name):
    """
    Load speaker metadata from a dataset.
    """
    metadata_file_path = hf_hub_download(
        repo_id=dataset_name,
        filename="speaker_metadata.csv",
        repo_type="dataset"
    )
    # print(f"File downloaded to: {metadata_file_path}")

    metadata_df = pd.read_csv(metadata_file_path)

    # map severity
    if 'severity_speech_impairment' in metadata_df.columns:
        metadata_df['severity'] = metadata_df['severity_speech_impairment'].apply(lambda x: x.split()[0].lower())
        metadata_df = metadata_df.drop(columns='severity_speech_impairment')



    return metadata_df

In [ ]:
# because the kernel is running remotely, the file is stored remotely as well
# this code helps to download it

from IPython.display import HTML
import base64

def create_download_link(filename):
    with open(filename, 'rb') as f:
        data = base64.b64encode(f.read()).decode()

    html = f'<a href="data:application/octet-stream;base64,{data}" download="{filename}">📥 Download {filename}</a>'
    display(HTML(html))

# # Usage
# create_download_link(filename)

# Load datasets

In [ ]:
dataset = load_dataset(DATASET_NAME, limit_to_30_seconds=LENGTH_LIMIT)
print(f"Loaded dataset with {len(dataset)} examples")

In [ ]:
metadata_df = load_speaker_metadata(DATASET_NAME)

# Load model and pipeline

In [ ]:
from transformers import pipeline

# Note: if handling data with more than 30 seconds, you need to set return_timestamps=True
# That will require batch size of 1 and also slow down the inference significantly due to timestamping

# Run on GPU if available, otherwise fall back to CPU.
# On GPU use float16: halves memory (whisper-large-v3 can OOM at batch 32 in float32) and is faster.
use_gpu = torch.cuda.is_available()
device = 0 if use_gpu else -1
torch_dtype = torch.float16 if use_gpu else torch.float32
print(f"Using device: {'cuda:0 (' + torch.cuda.get_device_name(0) + ')' if use_gpu else 'cpu'}, dtype: {torch_dtype}")

# Create pipeline
pipe = pipeline("automatic-speech-recognition",
                model=WHISPER_MODEL_NAME,
                device=device,
                torch_dtype=torch_dtype,
                #return_timestamps=False,
                )
print(f"Pipeline device: {pipe.device}")

# Run inference on dataset

In [ ]:
generate_kwargs={
    "language": LANGUAGE,
    "task": "transcribe",
    "max_length": 448, # Note: don't exceed 448 - otherwise you'll get index errors when max_length exceeds the models positional encoding limits
    "num_beams": 1,
    "do_sample": False
}
print(f"Using model: {pipe.model.name_or_path} with language: {LANGUAGE}")
N = len(dataset)
# N = 100
print(f"Number of examples in dataset: {N}")

results = []
for out in tqdm(pipe(AudioTextDataset(dataset.take(N)), batch_size=32, generate_kwargs=generate_kwargs), total=N):
    # print(out)
    results.append(out)

## Get overall results

In [ ]:
# Finalizing results...

print("Finalizing results...")
results_df = prepare_results(results)
print("Getting speaker metadata...")
results_df = add_speaker_metadata(results_df, metadata_df)
print(f"Calculating WER and CER for {len(results_df)} examples...")
results_df = calculate_error_rates(results_df, verbose=False)

In [ ]:
max_len=120
def truncate_text(text, max_length):
    if len(str(text)) > max_length:
        return str(text)[:max_length] + "..."
    return str(text)
df_display = results_df.copy()
df_display['prediction'] = df_display['prediction'].apply(
    lambda x: truncate_text(x, max_len)
)
df_display['ground_truth'] = df_display['ground_truth'].apply(
    lambda x: truncate_text(x, max_len)
)
df_display = df_display.drop(columns=['ground_truth_normalized', 'prediction_normalized'])
itables.show(df_display.round(2), connected=True)

## Get aggregated results

* we aggregate on the per speaker level first, and then can further aggregate by certain aspects

In [ ]:
per_speaker_agg = results_df[['speaker_id', 'severity', 'etiology', 'wer', 'cer']].groupby(['speaker_id', 'severity', 'etiology']).agg(['mean', 'count'])


In [ ]:
# per-speaker results
itables.show(per_speaker_agg.sort_values(by=['severity','speaker_id']).round(2), connected=True)

In [ ]:
# per-severity results
itables.show(per_speaker_agg.groupby(['severity']).agg(['mean', 'count']).round(2), connected=True)


In [ ]:
# per-etiology results
itables.show(per_speaker_agg.groupby(['etiology']).agg(['mean', 'count']).round(2), connected=True)

# Safe Predictions

In [ ]:
LOCAL_OUTPUT_FOLDER = '/tmp/predictions'
if not os.path.exists(LOCAL_OUTPUT_FOLDER):
    os.makedirs(LOCAL_OUTPUT_FOLDER)
print('saving predictions to folder: ', LOCAL_OUTPUT_FOLDER)

model_lower = WHISPER_MODEL_NAME.lower()
# check 'nonstandard' before 'standard' ('standard' is a substring of 'nonstandard')
if 'openai/whisper' in model_lower:
    stage_tag = 'baseline'
elif 'nonstandard' in model_lower:
    stage_tag = 'stage2'
elif 'standard' in model_lower:
    stage_tag = 'stage1'
else:
    stage_tag = 'other'
norm_tag = 'normalized' if APPLY_KY_NORMALIZATION else 'unnormalized'

output_filename = os.path.join(LOCAL_OUTPUT_FOLDER, f'{LANGUAGE}_{stage_tag}_{norm_tag}.tsv')
print('local output filename:', output_filename)
results_df.to_csv(output_filename, index=False, sep='\t', encoding='utf-8')

create_download_link(output_filename)
print("Click on this link to start download...")